In [ ]:
from pixel_arena.dataset_utils.coco import LABELS, integer_mask_to_pil 
import torch

# Re-index the original masks with the full label index

In [ ]:
all_valid_labels = list(filter(lambda x: x is not None, LABELS))
len(all_valid_labels)

In [ ]:
valid_label_idx_to_full_label_idx = torch.zeros(len(all_valid_labels) - 1, dtype=torch.uint8)

valid_label_idx = 0
for i in range(len(LABELS)):
    label = LABELS[i]
    if label == "other" or label is None:
        # skip "other" label
        # also skip removed labels (None)
        pass
    else:
        valid_label_idx_to_full_label_idx[valid_label_idx] = i
        valid_label_idx += 1

valid_label_idx_to_full_label_idx

In [ ]:
embedding_weight = valid_label_idx_to_full_label_idx.reshape(-1, 1)

In [ ]:
import os
import torch
from PIL import Image
from torchvision import transforms
from tqdm import tqdm

results_dir = "./results/coco/oneformer-150"
image_files = sorted([f for f in os.listdir(results_dir) if f.endswith("raw.png")])

tensors = []
mask_images = []
transform = transforms.PILToTensor()

for filename in tqdm(image_files):
    filepath = os.path.join(results_dir, filename)
    image = Image.open(filepath)
    tensor = transform(image)
    tensors.append(tensor)
    mask_images.append(image)

if tensors:
    all_mask_tensor = torch.stack(tensors)
    print(f"Stacked tensor shape: {all_mask_tensor.shape}")
else:
    print("No images found.")

In [ ]:
reindexed_mask_tensors = torch.nn.functional.embedding(all_mask_tensor.long(), embedding_weight).squeeze(-1)

## Check whether reindexed masks match the original masks

In [ ]:
import random
import matplotlib.pyplot as plt

In [ ]:
mask_idx = random.randint(0, all_mask_tensor.shape[0] - 1)

reindexed_mask_img = integer_mask_to_pil(reindexed_mask_tensors[mask_idx])
original_mask_img = mask_images[mask_idx]

# Show original and reindexed masks side by side
fig, axs = plt.subplots(1, 2, figsize=(8, 4))
axs[0].imshow(original_mask_img)
axs[0].set_title("Original Mask")
axs[0].axis("off")
axs[1].imshow(reindexed_mask_img)
axs[1].set_title("Reindexed Mask")
axs[1].axis("off")
plt.tight_layout()
plt.show()

## Export the reindexed masks

In [ ]:
for i in range(reindexed_mask_tensors.shape[0]):
    mask_img = integer_mask_to_pil(reindexed_mask_tensors[i])
    filename = image_files[i].replace("raw.png", "pred.png")
    mask_img.save(os.path.join(results_dir, filename))